# YOLO26 PPE: GitHub → Colab T4 → Google Drive
Đọc `docs/COLAB.md` trong repo. Chọn **Runtime → Change runtime type → GPU**. Sửa REPO_URL, REF và DATASET_ZIP. Dataset ZIP được chuẩn bị bằng `colab_utils.py pack`, không cần đưa ảnh lên GitHub. Dữ liệu và run được xử lý trên `/content`; chỉ đồng bộ kết quả sang Drive sau khi train xong. Mỗi lần train là run mới, không tự resume.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/phamminhhieupc/DEEPLERNING.git"
REF = "main"  # branch hoặc commit SHA
REPO_DIR = Path("/content/ppe-project")
if REPO_DIR.exists():
    raise FileExistsError("Repo đã tồn tại; dùng runtime mới hoặc kiểm tra thư mục trước khi clone lại")
subprocess.run(["git", "clone", "--branch", REF, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-dev.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch
import ultralytics
if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU trong Runtime > Change runtime type rồi chạy lại")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("torch:", torch.__version__, "Ultralytics:", ultralytics.__version__)
subprocess.run(["nvidia-smi"], check=True)


## Chuẩn bị dữ liệu trên ổ local
Upload `ppe_dataset_v2.zip` và `ppe_dataset_v2.zip.sha256` vào `MyDrive/PPE/datasets`. Dataset được giải nén vào `/content/ppe_dataset`, không đọc ảnh trực tiếp từ Drive. Cell không ghi đè dataset đã giải nén.

In [ ]:
from colab_utils import prepare_dataset
DATASET_ZIP = Path("/content/drive/MyDrive/PPE/datasets/ppe_dataset_v2.zip")
CHECKSUM_FILE = DATASET_ZIP.with_suffix(".zip.sha256")
LOCAL_DATASET = Path("/content/ppe_dataset")
if not DATASET_ZIP.is_file() or not CHECKSUM_FILE.is_file():
    raise FileNotFoundError("Upload ZIP và file SHA256 vào đúng đường dẫn trên Drive")
expected_hash = CHECKSUM_FILE.read_text().split()[0]
prepared = prepare_dataset(DATASET_ZIP, LOCAL_DATASET, expected_hash)
DATA_YAML = Path(prepared["data"])
import json
print(json.dumps(prepared, indent=2, ensure_ascii=False))


## Train
Mặc định là profile screening nhanh YOLO26n, 80 epoch, imgsz=640 để kiểm tra tốc độ và dữ liệu. Sau khi ổn định, đổi MODEL=yolo26s.pt, EPOCHS=200, PATIENCE=35, IMGSZ=960 cho train cuối. Hết VRAM: giảm batch hoặc dùng yolo26n.pt.


In [ ]:
## T4 profile: train trên local disk, sync Drive sau cùng
from datetime import datetime, timezone
from colab_utils import run_training
MODEL = "yolo26n.pt"
EPOCHS = 80
PATIENCE = 15
BATCH = 8  # nếu OOM: 4 rồi 2
IMGSZ = 640
WORKERS = 4
CACHE = "ram"  # thiếu RAM thì dùng "disk" hoặc "False"
FAST = True  # False nếu ưu tiên tái lập deterministic hơn tốc độ
PROJECT = Path("/content/runs_fast")
DRIVE_PROJECT = Path("/content/drive/MyDrive/PPE/runs")
RUN_NAME = "ppe_screening_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR = run_training(REPO_DIR, DATA_YAML, PROJECT, RUN_NAME, model=MODEL,
                       epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
                       imgsz=IMGSZ, workers=WORKERS, cache=CACHE,
                       fast=FAST, sync_to=DRIVE_PROJECT)


In [ ]:
from IPython.display import display, Image
summary = json.loads((RUN_DIR / "summary.json").read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("BEST MODEL:", summary["best_model"])
if (RUN_DIR / "results.png").is_file():
    display(Image(filename=str(RUN_DIR / "results.png")))
print("Log:", PROJECT / f"console_{RUN_NAME}.log")
